# 03 · Plan Explanation

> Part of the **ICAPS 2026 Planning Ontology Tutorial**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ai4society/ICAPS26-planning-ontology-tutorial/blob/main/notebooks/03_plan_explanation.ipynb)

A plan is a sequence of actions, but a user wants to know **why** those actions, in
that order. When the knowledge graph stores an explanation for the plan and for each
action, you can read them straight back out and assemble a narrative a person can
follow.

In this notebook you:

- **Read** the plan-level explanation, its cost, and its planner
- **List** the per-action explanations
- **Assemble** an ordered, step-by-step narrative of the plan

> **Tip.** The **Core** path reads the stored explanations as written. **Go deeper**
> grounds each template on the step's actual arguments.

---

## Setup

Run this first. In **Colab** it installs the dependencies and fetches the tutorial
data. **Locally** it uses your `requirements.txt` environment.

In [ ]:
# In Colab this installs dependencies and fetches the tutorial data.
# Locally it assumes you installed requirements.txt.
from pathlib import Path

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    import subprocess
    subprocess.run(["pip", "install", "-q", "rdflib", "pandas"], check=True)
    subprocess.run(["git", "clone", "-q",
                    "https://github.com/ai4society/ICAPS26-planning-ontology-tutorial.git"],
                   check=False)
    DATA = Path("ICAPS26-planning-ontology-tutorial/data")
else:
    DATA = Path("..") / "data"

print("data directory:", DATA.resolve())

---
## 1. Load the knowledge graph

The blocksworld KG holds a six-step plan for `problem_3_1`, with a plan-level
explanation, a per-action explanation on every action, and ordered steps. The steps
use a small tutorial extension (`tut:`) that numbers them and links each one to the
action it grounds.

In [ ]:
from rdflib import Graph, Namespace, Literal, RDF, RDFS

PO = Namespace("https://purl.org/ai4s/ontology/planning#")
TUT = Namespace("https://w3id.org/planning-ontology-tutorial#")
PREFIX = (
    "PREFIX po: <https://purl.org/ai4s/ontology/planning#> "
    "PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#> "
    "PREFIX tut: <https://w3id.org/planning-ontology-tutorial#> "
)

kg = Graph()
kg.parse(str(DATA / "kgs" / "blocksworld_tutorial.ttl"), format="turtle")
print(f"loaded blocksworld KG: {len(kg)} triples")

---
## 2. The plan-level explanation (Core)

Start with the big picture. One query returns the plan's cost, the planner that
generated it, and the stored `hasPlanExplanation` text.

In [ ]:
plan_query = PREFIX + """
SELECT ?cost ?planner ?explanation WHERE {
  po:problem_3_1 po:hasPlan ?plan .
  ?plan po:hasPlanCost ?cost .
  ?plan po:isGeneratedBy ?gen .
  ?gen rdfs:label ?planner .
  ?plan po:hasPlanExplanation ?explanation .
}
"""

row = list(kg.query(plan_query))[0]
print(f"planner : {row.planner}")
print(f"cost    : {int(row.cost)} steps")
print(f"why     : {row.explanation}")

---
## 3. The per-action explanations (Core)

Every action carries a `hasActionExplanation`: a short, reusable description of what
the action does. These read as templates, with variables such as `?x` and `?y`
standing in for the blocks.

In [ ]:
import pandas as pd

action_query = PREFIX + """
SELECT ?action ?explanation WHERE {
  ?a rdfs:label ?action .
  ?a po:hasActionExplanation ?explanation .
} ORDER BY ?action
"""

rows = [(str(r.action), str(r.explanation)) for r in kg.query(action_query)]
pd.DataFrame(rows, columns=["action", "explanation"])

---
## 4. Assemble the step-by-step narrative (Core)

Now join the two layers. The ordered steps come from the `tut:` extension, and each
step links to its action's explanation. Sorting by `tut:stepNumber` gives the plan
in execution order.

In [ ]:
narrative_query = PREFIX + """
SELECT ?n ?grounded ?explanation WHERE {
  ?step tut:stepOf po:plan_3_1 ;
        tut:stepNumber ?n ;
        rdfs:label ?grounded ;
        tut:groundsAction ?act .
  ?act po:hasActionExplanation ?explanation .
} ORDER BY ?n
"""

print("Plan for problem_3_1:\n")
for r in kg.query(narrative_query):
    print(f"  Step {int(r.n)}: {r.grounded}")
    print(f"          {r.explanation}\n")

---
## Go deeper: ground the templates on real arguments

The Core narrative shows the action templates verbatim, so `?x` and `?y` still
appear. To read as plain English, each template needs the step's actual blocks. The
step label (`stack b2 b1`) lists the arguments in order, and the action's parameters
(`?x - block`, `?y - block`) list the variables in order. Zip them together and
substitute.

In [ ]:
def parameters_in_order(graph, action_iri):
    q = PREFIX + """
    SELECT ?plabel WHERE { ?act po:hasParameter ?p . ?p rdfs:label ?plabel }
    ORDER BY ?p
    """
    labels = [str(r.plabel) for r in graph.query(q, initBindings={"act": action_iri})]
    return [lbl.split()[0] for lbl in labels]  # "?x - block" -> "?x"

def grounded_narrative(graph):
    q = PREFIX + """
    SELECT ?n ?grounded ?act ?explanation WHERE {
      ?step tut:stepOf po:plan_3_1 ;
            tut:stepNumber ?n ;
            rdfs:label ?grounded ;
            tut:groundsAction ?act .
      ?act po:hasActionExplanation ?explanation .
    } ORDER BY ?n
    """
    lines = []
    for r in graph.query(q):
        variables = parameters_in_order(graph, r.act)
        arguments = str(r.grounded).split()[1:]  # drop the action name
        text = str(r.explanation)
        for var, arg in zip(variables, arguments):
            text = text.replace(var, arg)
        lines.append((int(r.n), str(r.grounded), text))
    return lines

for n, grounded, text in grounded_narrative(kg):
    print(f"  {n}. {grounded}: {text}")

> **Note.** The grounded narrative is template-based, so it stays faithful to the
> KG. The same step records feed a natural-language generator or a large language
> model when you want richer prose.

---
## Recap and next

You pulled a plan's explanation, its action explanations, and its ordered steps from
the KG, then grounded the templates into a readable walk-through. Notebooks 01 to 03
cover the full path from ontology to applications.

| Next notebook | Focus |
| --- | --- |
| **00 · Quickstart** | Ask a planner-selection and an explanation question through one-line helpers |